In [1]:
! uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community

Using Python 3.13.14 environment at: C:\Code\End2endProject\.venv
Resolved 70 packages in 867ms
 Downloaded openai
Prepared 1 package in 2.46s
Installed 28 packages in 12.98s
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + attrs==26.1.0
 + flatbuffers==25.12.19
 + frozenlist==1.8.0
 + httpx-sse==0.4.3
 + jiter==0.17.0
 + langchain-classic==1.0.8
 + langchain-community==0.4.2
 + langchain-text-splitters==1.1.2
 + multidict==6.9.1
 + numpy==2.5.3
 + onnxruntime==1.30.0
 + openai==3.20.0
 + opencv-python==5.0.0.93
 + pillow==12.3.0
 + propcache==0.5.4
 + protobuf==7.36.2
 + pyclipper==1.4.0
 + pydantic-settings==2.15.0
 + rapidocr-onnxruntime==1.4.4
 + regex==2026.9.10
 + shapely==2.1.2
 + sqlalchemy==2.1.1
 + tiktoken==0.14.0
 + tqdm==4.70.1
 + yarl==1.25.1


In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
print(os.getcwd())

c:\Code\End2endProject\notebook


Data Ingestion

In [7]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader(r"C:\Code\End2endProject\data\AgenticAI.txt", encoding="utf-8")
documents = loader.load()

In [8]:
documents[0].page_content

'1. What is Agentic AI?\n\nAgentic AI is an AI system that can pursue a goal by reasoning about the task, deciding what actions to take, taking those actions, observing the results, and adapting its next actions accordingly.\n\nThe key idea is goal-oriented autonomous behavior.\n\nTraditional GenAI:\n\nInput → Generate response\n\nAgentic AI:\n\nGoal → Reason → Decide → Act → Observe → Adapt → Complete goal\n\n2. AI Agent\n\nAn AI agent is a system that perceives information from its environment and takes actions to achieve a given objective.\n\nConceptually:\n\nAgent = Perception + Reasoning + Decision-making + Action\n\nFor example:\n\nGoal: Resolve a customer\'s payment problem.\n\nThe agent may determine:\n\nIdentify the customer.\nCheck the order.\nCheck payment status.\nDetermine why payment failed.\nDecide what action is appropriate.\nTake the action.\nVerify the result.\n3. Main characteristics of Agentic AI\nAutonomy\n\nThe system can decide the next step without requiring the

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = text_splitter.split_documents(documents)
chunks[0].page_content

'1. What is Agentic AI?'

In [21]:
! uv pip install faiss-cpu

Using Python 3.13.14 environment at: C:\Code\End2endProject\.venv
Checked 1 package in 26ms


In [22]:
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [18]:
# Perform similarity search
query = "what is difference between Agentic AI and Generative AI"
docs = vectorstore.similarity_search(query, k=4)

for i, doc in enumerate(docs):
    print("document {i+1}:")
    print(doc.page_content)
    print("-"*50)

NameError: name 'vectorstore' is not defined

In [23]:
retriever = vectorstore.as_retriever()

NameError: name 'vectorstore' is not defined

In [24]:
from langchain_core.prompts import ChatPromptTemplate
template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [25]:
prompt=ChatPromptTemplate.from_template(template)

In [26]:
from langchain_core.output_parsers import StrOutputParser
output_parser=StrOutputParser()

In [27]:
from langchain.chat_models import ChatOpenAI
llm_model=ChatOpenAI(model_name="gpt-4o-mini")

ImportError: cannot import name 'ChatOpenAI' from 'langchain.chat_models' (c:\Code\End2endProject\.venv\Lib\site-packages\langchain\chat_models\__init__.py)

In [28]:
from langchain.schema.runnable import RunnablePassthrough


rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
)

ModuleNotFoundError: No module named 'langchain.schema'

In [ ]:
rag_chain.invoke("tell me about Agentic AI")

In [2]:
from google import genai
import os

from dotenv import load_dotenv
load_dotenv()

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

In [3]:
from google import genai
import os

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

print("Embedding models:")

for model in client.models.list():
    if "embedContent" in model.supported_actions:
        print(model.name)

Embedding models:
models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2
